In [2]:
# pip install torch torchvision torchaudio
# pip install git+https://github.com/facebookresearch/segment-anything-2.git
# mkdir -p checkpoints
# wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

In [3]:
import os
import cv2
import time
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import clip


In [4]:

# =============================
# CONFIGURATION
# =============================
INPUT_DIR = r"D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\DeshiFoodBD Dataset\food_data_english"
OUTPUT_DIR = r"D:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\DeshiFoodBD_english_SAM2_CLIP"
CHECKPOINT = "checkpoints/sam2_hiera_large.pt"
MODEL_CONFIG = "configs/sam2/sam2_hiera_l.yaml" # Built-in to the library
DEVICE = "cuda"

print(os.path.exists(INPUT_DIR))  # True or False?
os.makedirs(OUTPUT_DIR, exist_ok=True)



True


In [5]:

# =============================
# LOAD MODEL sam2
# =============================
print(f"Loading SAM 2 on {DEVICE}...")
sam2 = build_sam2(MODEL_CONFIG, CHECKPOINT, device=DEVICE, apply_postprocessing=True)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=16,          # High density to catch nested objects (rice vs bowl)
    points_per_batch=32,
    pred_iou_thresh=0.8,         # Reliability filter
    stability_score_thresh=0.92, # Quality filter
    box_nms_thresh=0.75,         # INCREASED: Allows rice to exist "inside" the bowl box
    min_mask_region_area=2000    # Deletes small crumbs, keeps full food items
)


Loading SAM 2 on cuda...


d:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\food-calorie-estimator\designing_nd_testing\segmentation_module\sam2\modeling\position_encoding.py:96: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  torch.arange(1, H + 1, dtype=torch.float32, device=device)


In [ ]:
# =============================
# LOAD CLIP MODEL
# =============================
print("Loading CLIP...")
clip_model, clip_preprocess = clip.load(r"./checkpoints/ViT-B-32.pt", device=DEVICE)

# REPLACE your labels with these
food_labels = [
    "a partial photo of food",
    "a partial photo of a bangladeshi meal",
    "A partial photo of a bakhar khana",
    "a partial photo of a meal",
    "a partial photo of rice",
    "a partial photo of curry",
    "a partial photo of vegetables",
    "a partial photo of cooked food",
    "a partial photo of a food dish",
    "a partial photo of fruit",
]

non_food_labels = [
    "a partial photo of non-food object",
    "a segmented photo of a steel object",
    "a partial photo of a soil object",
    "a partial photo of an plate",
    "a partial photo of a napkin",
    "a partial photo of a tablecloth",
    "a partial photo of a empty bowl",
    "a partial photo of a table surface",
    "a partial photo of a hand",
    "a partial photo of a background",
    "a partial photo of a spoon or fork",
    "a partial photo of human",
    "a partial photo of a finger"
    "a partial photo of a tool",
    "graphical image",
    "image of text",
    
]

all_labels = food_labels + non_food_labels

text_tokens = clip.tokenize(all_labels).to(DEVICE)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

def is_food_segment(image_rgb, mask_bool, threshold=0.65):
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        return False
    x0, x1 = xs.min(), xs.max() + 1
    y0, y1 = ys.min(), ys.max() + 1

    # Apply mask — black out everything outside the segment
    masked_rgb = image_rgb.copy()
    masked_rgb[~mask_bool] = 0
    crop = Image.fromarray(masked_rgb[y0:y1, x0:x1])  # ← only the segment

    image_input = clip_preprocess(crop).unsqueeze(0).to(DEVICE)
    binary_labels = ["a photo of food", "a photo of not food"]
    tokens = clip.tokenize(binary_labels).to(DEVICE)

    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features = clip_model.encode_text(tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]

    food_prob = probs[0].item()
    # print(f"  food_prob: {food_prob:.3f}")

    del image_input, image_features, probs
    torch.cuda.empty_cache()
    return food_prob > threshold

Loading CLIP...


In [7]:

# =============================
# HELPER: Save colored segment
# =============================
def save_colored_segment(image_rgb, mask, output_path):
    """
    Saves a cropped RGBA image of the segment, tightly bounded.
    The background is transparent; the object retains its original color.
    """
    # Build RGBA: use original color where mask is True, transparent elsewhere
    rgba = np.zeros((*image_rgb.shape[:2], 4), dtype=np.uint8)
    rgba[..., :3] = image_rgb
    rgba[..., 3] = (mask * 255).astype(np.uint8)

    # Tight crop around the masked region
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return
    x0, x1 = xs.min(), xs.max() + 1
    y0, y1 = ys.min(), ys.max() + 1
    cropped = rgba[y0:y1, x0:x1]

    Image.fromarray(cropped, mode='RGBA').save(output_path)


def deduplicate_masks(masks, iou_threshold=0.85):
    """Remove masks that heavily overlap with a larger mask already kept."""
    # Sort by area descending so we always keep the larger one first
    masks = sorted(masks, key=lambda m: m['area'], reverse=True)
    kept = []
    for candidate in masks:
        c_mask = candidate['segmentation']
        is_duplicate = False
        for kept_mask_data in kept:
            k_mask = kept_mask_data['segmentation']
            intersection = np.logical_and(c_mask, k_mask).sum()
            union = np.logical_or(c_mask, k_mask).sum()
            iou = intersection / union if union > 0 else 0
            if iou > iou_threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            kept.append(candidate)
    return kept




# =============================
# PROCESSING LOOP
# =============================
import shutil

total_start = time.time()

for root, dirs, files in os.walk(INPUT_DIR):
    # Mirror the subfolder structure in output
    rel_path = os.path.relpath(root, INPUT_DIR)
    output_root = os.path.join(OUTPUT_DIR, rel_path)
    os.makedirs(output_root, exist_ok=True)

    for filename in files:
        input_path = os.path.join(root, filename)
        output_path = os.path.join(output_root, filename)

        if filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg'):
            # Skip if already processed (resume support)
            base_name = os.path.splitext(filename)[0]
            existing = [f for f in os.listdir(output_root) if f.startswith(base_name + '_segment_')]
            if existing:
                print(f"Skipping already processed: {rel_path}/{filename} ({len(existing)} segments found)")
                continue

            # Process with SAM2
            image = cv2.imdecode(np.fromfile(input_path, dtype=np.uint8), cv2.IMREAD_COLOR)
            if image is None:
                print(f"Skipping unreadable file: {input_path}")
                shutil.copy2(input_path, output_path)
                continue

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            t0 = time.time()

            masks = mask_generator.generate(image_rgb)
            masks = deduplicate_masks(masks, iou_threshold=0.85)

            if masks:
                max_area = max(m['area'] for m in masks)
                masks = [m for m in masks if m['area'] >= 0.30 * max_area]

            masks = [m for m in masks if is_food_segment(image_rgb, m['segmentation'].astype(bool))]

            elapsed = time.time() - t0

            for i, mask_data in enumerate(masks):
                mask_bool = mask_data['segmentation'].astype(bool)
                segment_filename = f"{base_name}_segment_{i:03d}.png"
                save_colored_segment(
                    image_rgb, mask_bool,
                    os.path.join(output_root, segment_filename)
                )

            torch.cuda.empty_cache()  # Free GPU memory after each image
            print(f"[{elapsed:.2f}s] {rel_path}/{filename} -> {len(masks)} segments")

        else:
            # Copy non-JPG files as-is
            shutil.copy2(input_path, output_path)

total_elapsed = time.time() - total_start
print(f"\nAll done! Total time: {total_elapsed:.2f}s")

Skipping already processed: images\bakorkhani/000001.jpg (3 segments found)
Skipping already processed: images\bakorkhani/000002.jpg (21 segments found)
Skipping already processed: images\bakorkhani/000003.jpg (4 segments found)
Skipping already processed: images\bakorkhani/000004.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000007.jpg (3 segments found)
Skipping already processed: images\bakorkhani/000008.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000009.jpg (8 segments found)
Skipping already processed: images\bakorkhani/000010.jpg (5 segments found)
Skipping already processed: images\bakorkhani/000011.jpg (3 segments found)
Skipping already processed: images\bakorkhani/000012.jpg (4 segments found)
Skipping already processed: images\bakorkhani/000013.jpg (7 segments found)
Skipping already processed: images\bakorkhani/000016.jpg (7 segments found)
Skipping already processed: images\bakorkhani/000017.jpg (64 segments found)
Skipping a

d:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\food-calorie-estimator\designing_nd_testing\segmentation_module\sam2\sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (d:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\food-calorie-estimator\designing_nd_testing\segmentation_module\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(


[1.51s] images\bakorkhani/000027.jpg -> 0 segments
Skipping already processed: images\bakorkhani/000028.jpg (1 segments found)
Skipping already processed: images\bakorkhani/000029.jpg (3 segments found)
Skipping already processed: images\bakorkhani/000030.jpg (1 segments found)
Skipping already processed: images\bakorkhani/000031.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000032.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000035.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000037.jpg (4 segments found)
Skipping already processed: images\bakorkhani/000040.jpg (1 segments found)
Skipping already processed: images\bakorkhani/000041.jpg (1 segments found)
Skipping already processed: images\bakorkhani/000042.jpg (2 segments found)
Skipping already processed: images\bakorkhani/000043.jpg (6 segments found)
Skipping already processed: images\bakorkhani/000044.jpg (4 segments found)
Skipping already processed: images\ba

In [8]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

Free VRAM: 13.44 GB
